# Exploration des Données Amazon Reviews

Ce notebook explore le dataset Amazon Reviews v2 pour le moteur de recommandation e-commerce.

## Objectifs
- Comprendre la structure des données
- Analyser les statistiques descriptives
- Identifier les problématiques métier
- Préparer les données pour le modeling

In [ ]:
# Import des bibliothèques
import sys
sys.path.append('../src')

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from utils.spark_session import create_spark_session

# Créer la session Spark
spark = create_spark_session("exploration_amazon_reviews")

In [ ]:
# Configuration du schéma pour les données Amazon
schema = StructType([
    StructField("reviewerID", StringType(), True),
    StructField("asin", StringType(), True),
    StructField("reviewerName", StringType(), True),
    StructField("helpful", ArrayType(IntegerType()), True),
    StructField("reviewText", StringType(), True),
    StructField("overall", DoubleType(), True),
    StructField("summary", StringType(), True),
    StructField("unixReviewTime", LongType(), True),
    StructField("reviewTime", StringType(), True)
])

In [ ]:
# Charger les données Electronics
electronics_path = "../data/raw/amazon_reviews_electronics.json"

# Lecture du fichier JSON ligne par ligne
electronics_df = spark.read.json(electronics_path)

print(f"Nombre de reviews Electronics: {electronics_df.count():,}")
print(f"Colonnes: {electronics_df.columns}")

# Afficher le schéma
electronics_df.printSchema()

In [ ]:
# Aperçu des données
electronics_df.show(5, truncate=False)

In [ ]:
# Statistiques descriptives
print("=== Statistiques des notes ===")
electronics_df.select("overall").describe().show()

print("=== Distribution des notes ===")
electronics_df.groupBy("overall").count().orderBy("overall").show()

In [ ]:
# Analyse des utilisateurs et produits
print("=== Statistiques utilisateurs ===")
user_stats = electronics_df.groupBy("reviewerID").agg(
    count("*".alias("num_reviews")),
    avg("overall").alias("avg_rating")
)

user_stats.describe().show()

print("\n=== Statistiques produits ===")
product_stats = electronics_df.groupBy("asin").agg(
    count("*".alias("num_reviews")),
    avg("overall").alias("avg_rating")
    countDistinct("reviewerID").alias("num_reviewers")
)

product_stats.describe().show()

In [ ]:
# Convertir en Pandas pour visualisation
ratings_pd = electronics_df.select("overall").toPandas()

# Distribution des notes
plt.figure(figsize=(10, 6))
sns.countplot(data=ratings_pd, x='overall')
plt.title('Distribution des notes - Electronics')
plt.xlabel('Note')
plt.ylabel('Nombre de reviews')
plt.show()

In [ ]:
# Analyse temporelle
from pyspark.sql.functions import to_timestamp, year, month

# Convertir les timestamps
time_df = electronics_df.withColumn("review_timestamp", 
    to_timestamp(col("unixReviewTime"))
)

# Extraire année et mois
time_df = time_df.withColumn("year", year("review_timestamp"))
time_df = time_df.withColumn("month", month("review_timestamp"))

# Reviews par année
yearly_reviews = time_df.groupBy("year").count().orderBy("year")
yearly_reviews.show()

In [ ]:
# Analyse de la sparsité
total_users = electronics_df.select("reviewerID").distinct().count()
total_products = electronics_df.select("asin").distinct().count()
total_reviews = electronics_df.count()

possible_interactions = total_users * total_products
sparsity = 1 - (total_reviews / possible_interactions)

print(f"Nombre d'utilisateurs uniques: {total_users:,}")
print(f"Nombre de produits uniques: {total_products:,}")
print(f"Nombre total de reviews: {total_reviews:,}")
print(f"Nombre possible d'interactions: {possible_interactions:,}")
print(f"Sparsité: {sparsity:.6f} ({sparsity*100:.4}%)")

In [ ]:
# Identification des problématiques
print("=== Problématiques identifiées ===")
print("1. Cold Start Problem:")
print(f"   - Utilisateurs avec 1 review: {user_stats.filter(col('num_reviews') == 1).count():,}")
print(f"   - Produits avec 1 review: {product_stats.filter(col('num_reviews') == 1).count():,}")

print("\n2. Data Quality:")
null_counts = electronics_df.select([sum(col(c).isNull().cast("int")).alias(c) for c in electronics_df.columns])
null_counts.show()

## Conclusions de l'Exploration

### Points clés identifiés:
1. **Sparsité élevée**: Le dataset présente une sparsité typique des systèmes de recommandation
2. **Cold start**: Nombreux utilisateurs et produits avec peu d'interactions
3. **Distribution des notes**: Biais positif (plus de notes 4-5)
4. **Temporalité**: Les données couvrent plusieurs années

### Actions recommandées:
- Filtrer les utilisateurs/produits avec trop peu d'interactions
- Normaliser les notes pour gérer le biais
- Préparer les features pour le collaborative filtering
- Planifier la gestion du cold start